# Visualisation of EEG & Sleep scoring

### Import recordings

Load packages

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import signal
import os
from IPython.display import display
from ipyfilechooser import FileChooser
import ipywidgets as widgets
from ephyviewer import mkQApp, MainViewer, TraceViewer
import ephyviewer
from scipy.stats import zscore
from itertools import groupby
from ephyviewer import mkQApp, MainViewer, TraceViewer, TimeFreqViewer
from ephyviewer import InMemoryAnalogSignalSource
from ephyviewer import mkQApp, MainViewer, TraceViewer, CsvEpochSource, EpochEncoder
from ephyviewer import mkQApp, MainViewer, EpochViewer,EventList
from ephyviewer import InMemoryEventSource, InMemoryEpochSource
import json
import IPython
from matplotlib import cm
from matplotlib.colors import to_hex
import ast
from ephyviewer import AnalogSignalSourceWithScatter
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.colors as mcolors
from scipy import stats
import re
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta
from datetime import datetime, timezone
from zoneinfo import ZoneInfo  # Only in Python 3.9+
import shutil
import mne
from PyQt5 import QtCore
from ephyviewer import SpikeTrainViewer, InMemorySpikeSource
import scipy.io
import pandas as pd
import numpy as np

%matplotlib widget

Choose .edf file

In [ ]:
file= 'AZSA10-E_12112018' #AGAL06-E_06032017 #GUNO03-E_24042018 #AZSA10-E_12112018

In [ ]:
dpath = f"C:/Users/Manip2/Documents/EpiKid/ScoredSTS/{file}.edf"
folder_base = Path(dpath) 

Load STS file & scoring infos

In [ ]:
txtt = f"C:/Users/Manip2/Documents/EpiKid/ScoredSTS/{file}.txt"

with open(txtt, encoding="utf_8", errors='ignore') as f:
    contents = f.read()
keep_words = ["Stage N1  ", "Stage N2  ", "Stage N3  ", "REM  ", "WAKE  "]
s = contents
matches = []
i = 0
while i < len(s):
    for w in keep_words:
        if s.startswith(w, i):
            matches.append(w)
            i += len(w)
            break
    else:
        i += 1  # move forward if no match

Load TSV files from Snooz 

In [ ]:
import csv
with open(f"C:/Users/Manip2/Documents/EpiKid/ScoredSTS/{file}_repaired.tsv") as fd:
    rd = csv.reader(fd, delimiter="\t", quotechar='"')
    for row in rd:
        #print(row)

Example of sleep stage scoring from Snooz

In [ ]:
import csv
with open(f"C:/Users/Manip2/Documents/EpiKid/SnoozTutorialFiles/snooz_sleep_stages.tsv") as fd:
    rd = csv.reader(fd, delimiter="\t", quotechar='"')
    for row in rd:
        print(row)

Clean corrupted edf file (no montage)

In [ ]:
raw = mne.io.read_raw_edf(
    dpath,
    preload=True,
    verbose="error", 
    encoding='latin1'
)

raw.set_meas_date(None)
raw.set_annotations(mne.Annotations([], [], []))
raw.rename_channels(
    lambda ch: ch[:16].replace(" ", "_")
)
raw.pick_types(
    eeg=True,
    eog=True,
    emg=True,
    ecg=True,
    misc=True
)


data = raw.get_data()
info = raw.info.copy()

raw_clean = mne.io.RawArray(data, info)
mne.export.export_raw(
    f'{Path(dpath).parent}/{Path(dpath).name[:-4]}_repaired.edf',
    raw_clean,
    fmt="edf",
    physical_range=(-1000, 1000),
    overwrite=True
)
# reload to check
data = mne.io.read_raw_edf(f'{Path(dpath).parent}/{Path(dpath).name[:-4]}_repaired.edf', preload=False)
print(data)

Load metadata of edf

In [ ]:
#data = mne.io.read_raw_edf(dpath, encoding='latin1')

raw_data = data.get_data()
# you can get the metadata included in the file and a list of all channels:
info = data.info
channels = data.ch_names
timestamps = data.times
duration = data.duration
samplerate = data.info.get('sfreq')  # in Hz

print(data.info.get('meas_date'))
data.info.get('subject_info')
lowpass=data.info.get('lowpass')
highpass=data.info.get('highpass')

combined = raw_data.T
print(f'{round(np.shape(raw_data)[1]/samplerate/60/60,3)}h recording if sampled at {samplerate}Hz ({duration} seconds)')
print(f'{np.shape(raw_data)[0]} channels found: {channels}')

In [ ]:
print('Does the number of detected epoch match the length of the signal?', np.floor(duration/30) == len(matches))
SleepScoring= pd.DataFrame(matches, columns=['label'])
SleepScoring['time'] = list(range(0, len(SleepScoring) * 30, 30))
SleepScoring['duration'] = 30
# correct if not matching total recording time
SleepScoring.loc[(SleepScoring.shape[0]-1), 'duration']= duration - (SleepScoring.loc[(SleepScoring.shape[0]-2), 'time'] +30)
AutoSleepScoring_filename = os.path.join(Path(f'{Path(txtt).parent}'), f'EphyViewer_approxScoring_{file}.csv')
SleepScoring.to_csv(AutoSleepScoring_filename, index=False)

Spikes=pd.DataFrame(columns=['label', 'time', 'duration'])
Spikes_filename = os.path.join(Path(f'{Path(txtt).parent}'), f'EphyViewer_Spikes_{file}.csv')
Spikes.to_csv(Spikes_filename, index=False)


Do montage

In [ ]:
montage = 'Fp1-C3,C3-T3,T3-O1,Fp2-C4,C4-T4,T4-O2,EMG1+,EMG2+' if file=='GUNO03-E_24042018' else 'Fp2-F4,F4-C4,C4-P4,P4-O2,Fp1-F3,F3-C3,C3-P3,P3-O1,Fz-Cz,Cz-Pz,Fp2-F8,F8-T4,T4-T6,T6-O2,Fp1-F7,F7-T3,T3-T5,T5-O1,EMG1+,EMG2+' #AL
channels_clean = [ch.replace('EEG ', '') for ch in channels]
channels_clean = [ch.replace('EEG_', '') for ch in channels_clean]

pairs = montage.split(',')
montage_eeg=[]
montage_name=[]
for pair in pairs:
    if pair.count('-')==0:
        ch1 = pair
        idx1 = channels_clean.index(ch1)
        eeg=combined[:,idx1]
        montage_eeg = zscore(eeg[:,np.newaxis]) if len(montage_eeg) == 0 else np.append(montage_eeg, zscore(eeg[:,np.newaxis]), axis=1)
        montage_name.append(pair)
    else:        
        ch1, ch2 = pair.split('-')
        idx1 = channels_clean.index(ch1)
        idx2 = channels_clean.index(ch2)
        eeg=combined[:,idx2]-combined[:,idx1]
        nyq = samplerate / 2
        f_lowcut = 0.5
        f_hicut = 50.0 #70
        Wn = [f_lowcut/nyq, f_hicut/nyq]
        sos = signal.butter(6, Wn, btype='band', output='sos')
        eeg = signal.sosfiltfilt(sos, eeg)
        montage_eeg = zscore(eeg[:,np.newaxis]) if len(montage_eeg) == 0 else np.append(montage_eeg, zscore(eeg[:,np.newaxis]), axis=1)
        montage_name.append(pair)
eeg=[]
eeg_name=[]
for ch in channels_clean:
    idx1 = channels_clean.index(ch)
    eeg1=combined[:,idx1]
    eeg = zscore(eeg1[:,np.newaxis]) if len(eeg) == 0 else np.append(eeg, zscore(eeg1[:,np.newaxis]), axis=1)
    eeg_name.append(ch)

Check all channels (no montage)

In [ ]:
app = mkQApp()
winlen = 10 # default window length in sec
t_start = 0.

win = MainViewer(debug=False, show_auto_scale=True)
view1 = TraceViewer.from_numpy(eeg, samplerate, t_start, 'Signals', channel_names=eeg_name)

cmap = cm.Spectral
values = np.linspace(0, 1, np.array(eeg).shape[1])
colormap = [to_hex(rgb) for rgb in cmap(values)]
for idx in range(np.array(eeg).shape[1]): 
    view1.by_channel_params[f'ch{idx}', 'color'] = colormap[idx]

view1.params['display_labels'] = True
view1.params['scale_mode'] = 'same_for_all'
view1.auto_scale()
view1.params['xsize'] = winlen
win.add_view(view1)
win.navigation_toolbar.spinbox_xsize.setValue(winlen)

#Run
win.show()
app.exec()


With sleep scoring (montage)

In [ ]:
labels = ["WAKE  ", "Stage N1  ", "Stage N2  ", "Stage N3  ", "REM  ", ]

epoch_dur = 30 # define epoch duration in sec
winlen = 30 # default window length in sec

SleepScoring_filename = f'{Path(txtt).parent}/EphyViewer_approxScoring_{file}.csv'
#SleepScoring_filename = f'{dpath}/Sleep_Scoring_{len(labels)}Stages_{epoch_dur}sEpoch.csv'
source_epoch = CsvEpochSource(SleepScoring_filename, labels)

#you must first create a main Qt application (for event loop)
app = mkQApp()

t_start = 0.

#Create the main window that can contain several viewers
win = MainViewer(debug=False, show_auto_scale=True)

#create a viewer for signal
source =InMemoryAnalogSignalSource(montage_eeg, samplerate, t_start, channel_names=montage_name)
view1 = TraceViewer(source=source)

view1.params['xsize']= winlen
view1.params['display_labels'] = True
view1.params['background_color'] = '#FFFFFF'
view1.params['label_fill_color'] = '#FFFFFF'
view1.params['vline_color'] = "#FFFFFF00"

view1.params['scale_mode'] = 'same_for_all'
colormap = ["#000000"]* 50
for idx, ch in enumerate(montage_name): 
    view1.by_channel_params[f'ch{idx}', 'color'] = colormap[idx] #FF0000 red, #00FF00 green, and #0000FF blue
view1.auto_scale()

#create a viewer for the encoder itself
view2 = EpochEncoder(source=source_epoch, name='Sleep Scoring')

view2.params['xsize'] = winlen
view2.params['new_epoch_step'] = epoch_dur

view2.by_label_params['label0', 'color'] = "#09ff00" 
view2.by_label_params['label1', 'color'] = "#00f7ff"
view2.by_label_params['label2', 'color'] = "#fffb00"
view2.by_label_params['label3', 'color'] = "#ff00d4"
view2.by_label_params['label4', 'color'] = "#ff0000"
view2.params['background_color'] = '#FFFFFF'
view2.params['label_fill_color'] = '#FFFFFF'
view2.params['view_mode'] = 'flat'
view2.params['vline_color'] = "#FFFFFF00"

view2.controls.hide()



# FFT
view3 = TimeFreqViewer(source=source, name='FFT')

view3.params['show_axis'] = True
view3.params['timefreq', 'f_start'] = 1
view3.params['timefreq', 'f_stop'] = 50
view3.params['timefreq', 'deltafreq'] = 1 #interval in Hz
view3.params['xsize'] = winlen
for idx, ch in enumerate(montage_name):
    """
    if ch == 'EMG': 
        view3.by_channel_params[f'ch{idx}', 'visible'] = False
    else:        
        view3.by_channel_params[f'ch{idx}', 'clim'] = 1
        view3.by_channel_params[f'ch{idx}', 'visible'] = True
    """
    if ch == 'C3-P3': 
        view3.by_channel_params[f'ch{idx}', 'clim'] = 1
        view3.by_channel_params[f'ch{idx}', 'visible'] = True
    else: 
        view3.by_channel_params[f'ch{idx}', 'visible'] = False



win.add_view(view1)
win.add_view(view2)
#win.add_view(view3)
win.navigation_toolbar.spinbox_xsize.setValue(winlen)
win.show()

app.exec()

# press '1', '2', '3', '4' etc, to encode state.
# or toggle 'Time range selector' and then use 'Insert within range'

Annotate spikes as epoch on montage

In [ ]:
labels = ["WAKE  ", "Stage N1  ", "Stage N2  ", "Stage N3  ", "REM  ", ]
labels2 = [f"{i}_{s}" for i, s in enumerate(montage.split(','), start=1) if "EMG" not in s]

epoch_dur = 30 # define epoch duration in sec
spike_dur = .01 # define epoch duration in sec
winlen = 30 # default window length in sec

SleepScoring_filename = f'{Path(txtt).parent}/EphyViewer_approxScoring_{file}.csv'
#SleepScoring_filename = f'{dpath}/Sleep_Scoring_{len(labels)}Stages_{epoch_dur}sEpoch.csv'
source_epoch = CsvEpochSource(SleepScoring_filename, labels)

Spikes_filename = f'{Path(txtt).parent}/EphyViewer_Soikes_{file}.csv'
#SleepScoring_filename = f'{dpath}/Sleep_Scoring_{len(labels)}Stages_{epoch_dur}sEpoch.csv'
source_epoch2 = CsvEpochSource(Spikes_filename, labels2)


#you must first create a main Qt application (for event loop)
app = mkQApp()

t_start = 0.

#Create the main window that can contain several viewers
win = MainViewer(debug=False, show_auto_scale=True)

#create a viewer for signal
source =InMemoryAnalogSignalSource(montage_eeg, samplerate, t_start, channel_names=montage_name)
view1 = TraceViewer(source=source)

view1.params['xsize']= winlen
view1.params['display_labels'] = True
view1.params['background_color'] = "#FFFFFF"
view1.params['label_fill_color'] = '#FFFFFF'
view1.params['vline_color'] = "#000000"

view1.params['scale_mode'] = 'same_for_all'
colormap = ["#000000"]* 50
for idx, ch in enumerate(montage_name): 
    view1.by_channel_params[f'ch{idx}', 'color'] = colormap[idx] #FF0000 red, #00FF00 green, and #0000FF blue
view1.auto_scale()

#create a viewer for the encoder itself
view2 = EpochEncoder(source=source_epoch, name='Sleep Scoring')

view2.params['xsize'] = winlen
view2.params['new_epoch_step'] = epoch_dur

view2.by_label_params['label0', 'color'] = "#09ff00" 
view2.by_label_params['label1', 'color'] = "#00f7ff"
view2.by_label_params['label2', 'color'] = "#fffb00"
view2.by_label_params['label3', 'color'] = "#ff00d4"
view2.by_label_params['label4', 'color'] = "#ff0000"

view2.by_label_params['label0', 'key'] = '9'
view2.by_label_params['label1', 'key'] = '9'
view2.by_label_params['label2', 'key'] = '9'
view2.by_label_params['label3', 'key'] = '9'
view2.by_label_params['label4', 'key'] = '9'

view2.params['background_color'] = '#FFFFFF'
view2.params['label_fill_color'] = "#FFFFFF"
view2.params['vline_color'] = "#000000"

view2.params['view_mode'] = 'flat'
view2.controls.hide()



#create a viewer for the encoder itself
view4 = EpochEncoder(source=source_epoch2, name='Spikes')
view4.params['xsize'] = winlen
view4.params['new_epoch_step'] = spike_dur
view4.params['background_color'] = '#FFFFFF'
view4.params['label_fill_color'] = "#FFFFFF"
view4.params['vline_color'] = "#000000"

colormap = ["#000000"]* 50
for idx, ch in enumerate(labels2): 
    view4.by_label_params[f'label{idx}', 'color'] = colormap[idx] #FF0000 red, #00FF00 green, and #0000FF blue
view4.controls.hide()


win.add_view(view1)
win.add_view(view4)
win.add_view(view2)
win.navigation_toolbar.spinbox_xsize.setValue(winlen)
win.show()

app.exec()

# press '1', '2', '3', '4' etc, to encode state.
# or toggle 'Time range selector' and then use 'Insert within range'

## Display detected interictal spikes

Load .mat file

In [ ]:
mat = scipy.io.loadmat(f"C:/Users/Manip2/Documents/EpiKid/ScoredSTS/spikes_{file}.mat", squeeze_me=True, struct_as_record=False)
data = mat["spikes"]
result = {}
for s in data:
    times = np.asarray(s.time).flatten()
    value_name = str(np.asarray(s.value).flatten()[0]).replace("EEG ", "")

    if value_name not in result:
        result[value_name] = []
    result[value_name].extend(times.tolist())
ordered_result = {k: result[k] for k in channels_clean if k in result}

With sleep scoring + spikes (no montage)

In [ ]:
labels = ["WAKE  ", "Stage N1  ", "Stage N2  ", "Stage N3  ", "REM  ", ]

epoch_dur = 30 # define epoch duration in sec
winlen = 30 # default window length in sec

scatter_indexes = {
    i: pd.Series(np.asarray(v).ravel() * samplerate, name="start time", dtype=int)
    for i, v in enumerate(ordered_result.values())
}

order_index = {k: i for i, k in enumerate(channels_clean)}

scatter_channels = {
    idx: [order_index[k]]
    for idx, k in enumerate(ordered_result.keys())
    if k in order_index
}

SleepScoring_filename = f'{Path(txtt).parent}/EphyViewer_approxScoring_{file}.csv'
source_epoch = CsvEpochSource(SleepScoring_filename, labels)

#you must first create a main Qt application (for event loop)
app = mkQApp()

t_start = 0.

#Create the main window that can contain several viewers
win = MainViewer(debug=False, show_auto_scale=True)

#create a viewer for signal
source = AnalogSignalSourceWithScatter(eeg, samplerate, t_start, scatter_indexes, scatter_channels, channel_names=eeg_name)
view1 = TraceViewer(source=source)

view1.params['xsize']= winlen
view1.params['display_labels'] = True
view1.params['background_color'] = '#FFFFFF'
view1.params['label_fill_color'] = '#FFFFFF'

view1.params['scale_mode'] = 'same_for_all'
colormap = ["#000000"]* 50
for idx, ch in enumerate(eeg_name): 
    view1.by_channel_params[f'ch{idx}', 'color'] = colormap[idx] #FF0000 red, #00FF00 green, and #0000FF blue
view1.auto_scale()

#create a viewer for the encoder itself
view2 = EpochEncoder(source=source_epoch, name='Sleep Scoring')

view2.params['xsize'] = winlen
view2.params['new_epoch_step'] = epoch_dur

view2.by_label_params['label0', 'color'] = "#09ff00" 
view2.by_label_params['label1', 'color'] = "#00f7ff"
view2.by_label_params['label2', 'color'] = "#fffb00"
view2.by_label_params['label3', 'color'] = "#ff00d4"
view2.by_label_params['label4', 'color'] = "#ff0000"
view2.params['background_color'] = '#FFFFFF'
view2.params['label_fill_color'] = '#FFFFFF'
view2.params['view_mode'] = 'flat'
view2.controls.hide()


win.add_view(view1)
win.add_view(view2)
win.navigation_toolbar.spinbox_xsize.setValue(winlen)
win.show()

app.exec()

# press '1', '2', '3', '4' etc, to encode state.
# or toggle 'Time range selector' and then use 'Insert within range'
